<span style="color:red;font-size:2em;font-weight:bold"> PARTIE 2 - Tracking des expérimentations via MLFlow</span>

<span style="color:blue;font-size:1.5em;font-weight:bold;background-color:yellow"> Modules </span>

In [ ]:
# Module pour recharger un module sans redemarrer le kernel
# import importlib
%load_ext autoreload
%autoreload 2

In [ ]:
# Roots
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path

# Stats
# from scipy.stats import zscore, chi2_contingency, f_oneway, chi2

#Selection
from sklearn.model_selection import (
    train_test_split,
    GridSearchCV, 
    cross_validate,
    StratifiedShuffleSplit,
    cross_val_predict,
    KFold,
    StratifiedKFold,
)
# Metrics
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, f1_score, fbeta_score, precision_recall_curve, 
    precision_score, recall_score,
    roc_curve, ConfusionMatrixDisplay, PrecisionRecallDisplay,
    RocCurveDisplay,make_scorer
)

# # Feature importance
# from sklearn.inspection import permutation_importance

# #Preprocess
# from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
# from sklearn.preprocessing import (
#     OneHotEncoder, StandardScaler, FunctionTransformer, 
#     RobustScaler,PowerTransformer
# )
# from imblearn.pipeline import Pipeline
# from imblearn.over_sampling import SMOTE
# from imblearn.under_sampling import RandomUnderSampler

#Modèles
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier,
    HistGradientBoostingClassifier
)
from catboost import CatBoostClassifier, Pool

In [ ]:
# Sert à éviter les Warnings avec les transformations sur des vues en transformant 
# ces warning en erreur obligeant ainsi à ne travailler que sur des copies ou les originaux.

pd.set_option('mode.chained_assignment','raise')

In [ ]:
# Ajoute le dossier datas_manipulation au sys.path. Remarque ne pas oublier le __init__.py dans le dossier datas_manipulation
import sys
# root_path = Path(__file__).resolve().parents[1] # Ne fonctionne pas sur notebook
root_path = Path.cwd().parent
sys.path.append(str(root_path))

In [ ]:
# Fonctions personnelles

# utils
from notebooks.utils.features_type_list import features_type
from notebooks.utils.metrics_classification import top_score
from notebooks.models_tools.model_attributes import model_attr
# pipeline
from notebooks.models_tools.pipeline_builder import build_classification_pipeline

# préprocessing
from notebooks.models_tools.preprocessing import preproc_numerical_features,build_preprocessor
# modeling
from notebooks.models_tools.models_runner import modeling_cv, predict_models_cv
from notebooks.plotting.make_model_plots import get_feature_importance, plot_feature_importance, plot_hyperparam_effect

In [ ]:
# Paramètres globaux

# Création dossier results
save_path = root_path.joinpath('datas/results')
Path.mkdir(save_path,exist_ok = True)

#
random_state=42
cv=5

<span style="color:blue;font-size:1.5em;font-weight:bold;background-color:yellow"> Datasets </span>

In [ ]:
# Chemin du dataset d'entrainement/test du modèle
datas_path = (
    root_path /'datas'/'raw_datas'/
    'Projet+Mise+en+prod+-+home-credit-default-risk'/'final_datasets'
)

In [ ]:
# Importation de la donnée
Xy= pd.read_parquet(datas_path/"train.parquet")
Xy.head()

In [ ]:
# Définition des prédicteurs X et de la cible y
X = Xy.drop(columns=['TARGET'])
y = Xy['TARGET']

In [ ]:
# Identification des features numériques et catégorielles
numeric_list, cat_list = features_type(X)

In [ ]:
# Dictionnaire de modèles
models_full = {
    # défaut prior: inutilisable car prédit uniquement la classe majoritaire
    'dummy': DummyClassifier(strategy='stratified'),
    # defaut lbfgs: supporte mal le déséquilibre des classes
    'lr': LogisticRegression(
        random_state=random_state,
        solver='liblinear'
    ),
    'rf': RandomForestClassifier(
        n_estimators=100, #defaut = 100
        random_state=random_state,#defaut = None
    ),
    
    'gb': GradientBoostingClassifier(
        n_estimators=100,#defaut = 100
        random_state=random_state,#defaut = None
    ),
    # Similaire a gb (mais plus perf pour gros data) et inspiré de Light GBM
    'hgb': HistGradientBoostingClassifier(
        max_iter=100,# remplace n_estimators, defaut = 100
        random_state= random_state
    ),
    'cb': CatBoostClassifier(
        random_state=random_state,#defaut = None
        # Evite de printer 1 milliard de ligne de progression
        logging_level='Silent'
    )
}

<span style="color:orange;font-weight:bold"> GrabdientBoosting est similaire au XGBoost mais moins performant (arbres construit par niveau, lent sur gros datasets, peu d'options avancées) que XGBoost (Régulariation L1/L2, gestion native des valeurs manquantes, early stoppping avancée, parralélisme efficace...). De même HistoGradientBoosting est similaire a LightGBM avec un splitting en histogramme, une efficacité sur gros datasets et la gestion des valeurs manquantes. Dans les deux cas, les modèles sont issus de sklearn donc plus simple a implémenter mais moins performants in fine. </span>

In [ ]:
scoring = {
        'f2':make_scorer(fbeta_score, beta=2),
        'prec':'precision',
        'recall':'recall',
    }